# Vast.ai: first-four-minute harmony feature extraction

This notebook creates one 45-feature harmony row for each of the 7,324 selected tracks. It is configured for the 12-thread Vast.ai offer and deliberately keeps notebook output quiet.

Outputs:

- `harmony_features_checkpoint.csv`: resumable detailed checkpoint
- `harmony_features_raw.csv`: final detailed extraction table
- `harmony_df.csv`: clean `TRACK_ID + 45 features` table
- `harmony_feature_errors.csv`: failures, if any
- `harmony_extraction_config.json`: frozen settings

The extraction uses the first `min(track duration, 240 seconds)`, Librosa CQT chroma, Tonnetz summaries, 10 CPU workers, bounded 50-track batches, and Drive backup every 250 completed tracks.


## 1. Install dependencies quietly

Run once per new instance. CUDA is not used by this extraction.


In [ ]:
%pip install -q --disable-pip-version-check "numpy<3" "pandas<3" "librosa==0.11.0" "soundfile>=0.12" "joblib>=1.4" "tqdm>=4.66"


In [ ]:
import os

# Prevent every worker from creating its own BLAS thread pool.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from pathlib import Path
import gc
import hashlib
import json
import math
import re
import shutil
import subprocess
import time
import warnings

import librosa
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=UserWarning)
print("Runtime ready. CPU threads:", os.cpu_count())


## 2. Configuration


In [ ]:
PROJECT_ROOT = Path("/workspace/MTG_Jamendo_7324_full_quality")
AUDIO_ROOT = PROJECT_ROOT / "audio"
OUTPUT_ROOT = PROJECT_ROOT / "harmony_features"
CSV_PATH = PROJECT_ROOT / "split_csv.csv"

REMOTE_DATA = "gdrive:music-genre-classification/data"
REMOTE_AUDIO = "gdrive:MTG_Jamendo_7324_full_quality/audio"

CHECKPOINT_CSV = OUTPUT_ROOT / "harmony_features_checkpoint.csv"
FINAL_RAW_CSV = OUTPUT_ROOT / "harmony_features_raw.csv"
FINAL_CLEAN_CSV = OUTPUT_ROOT / "harmony_df.csv"
ERROR_CSV = OUTPUT_ROOT / "harmony_feature_errors.csv"
CONFIG_JSON = OUTPUT_ROOT / "harmony_extraction_config.json"
RCLONE_LOG = PROJECT_ROOT / "rclone_audio_copy.log"

SAMPLE_RATE = 16_000
HOP_LENGTH = 512
MAX_DURATION_SECONDS = 240.0
N_CHROMA = 12
BINS_PER_OCTAVE = 36
N_OCTAVES = 7
RELATIVE_RMS_THRESHOLD = 0.01
ABSOLUTE_RMS_THRESHOLD = 1e-6

N_WORKERS = min(10, max(1, (os.cpu_count() or 1) - 2))
PARALLEL_BATCH_SIZE = 50
CHECKPOINT_EVERY_BATCH = True
DRIVE_BACKUP_EVERY = 250
RETRY_FAILED_TRACKS = True
LIMIT_TRACKS = None

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

PITCH_NAMES = ["c", "csharp", "d", "dsharp", "e", "f", "fsharp", "g", "gsharp", "a", "asharp", "b"]
FEATURE_COLUMNS = (
    [f"chroma_{name}_mean" for name in PITCH_NAMES]
    + [f"chroma_{name}_std" for name in PITCH_NAMES]
    + [f"tonnetz_{i:02d}_mean" for i in range(1, 7)]
    + [f"tonnetz_{i:02d}_std" for i in range(1, 7)]
    + [
        "tonal_concentration_mean", "tonal_concentration_std",
        "chroma_entropy_mean", "chroma_entropy_std",
        "chroma_flux_mean", "chroma_flux_std",
        "tonnetz_movement_mean", "tonnetz_movement_std",
        "valid_tonal_ratio",
    ]
)
assert len(FEATURE_COLUMNS) == 45
print("Workers:", N_WORKERS, "| Features:", len(FEATURE_COLUMNS))


## 3. Quietly copy the manifest and audio to local SSD

This cell writes rclone details to a log file and prints only a final summary. It is safe to rerun. If the audio is already local, rclone checks and skips matching files.

While it runs, monitor from a separate Vast terminal without stressing Jupyter:

```bash
tail -n 12 /workspace/MTG_Jamendo_7324_full_quality/rclone_audio_copy.log
find /workspace/MTG_Jamendo_7324_full_quality/audio -type f -name '*.mp3' | wc -l
du -sh /workspace/MTG_Jamendo_7324_full_quality/audio
```


In [ ]:
def run_quiet_rclone(arguments, log_path=RCLONE_LOG):
    command = ["rclone", *arguments, "--log-file", str(log_path), "--log-level", "NOTICE", "--stats", "5m", "--stats-one-line"]
    completed = subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if completed.returncode != 0:
        tail = log_path.read_text(errors="replace").splitlines()[-20:] if log_path.is_file() else []
        raise RuntimeError("rclone failed:\n" + "\n".join(tail))


usage = shutil.disk_usage("/workspace")
assert usage.free >= 80 * 2**30, f"Need at least 80 GiB free; found {usage.free / 2**30:.1f} GiB"

print("Starting quiet rclone copy. Progress is in:", RCLONE_LOG)
started = time.time()
run_quiet_rclone(["copyto", f"{REMOTE_DATA}/split_csv.csv", str(CSV_PATH)])
run_quiet_rclone(["copy", REMOTE_AUDIO, str(AUDIO_ROOT), "--transfers", "8", "--checkers", "16"])
mp3_count = sum(1 for _ in AUDIO_ROOT.rglob("*.mp3"))
print(f"Copy check finished in {(time.time()-started)/3600:.2f} h | MP3 files: {mp3_count:,}")
assert mp3_count == 7_324


## 4. Load the selected tracks and resolve audio paths


In [ ]:
def canonical_track_id(value):
    text = str(value).strip()
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    text = re.sub(r"^track[_\-\s]*", "", text, flags=re.IGNORECASE)
    return str(int(text)) if re.fullmatch(r"\d+", text) else None


metadata = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
assert "TRACK_ID" in metadata and "PATH" in metadata
metadata["_key"] = metadata["TRACK_ID"].map(canonical_track_id)
assert len(metadata) == 7_324 and metadata["_key"].is_unique

def resolve_audio(row):
    relative = str(row["PATH"]).replace("\\", "/").lstrip("/")
    candidate = AUDIO_ROOT / relative
    if candidate.is_file():
        return str(candidate)
    numeric = row["_key"]
    alternatives = [
        AUDIO_ROOT / numeric[-2:] / f"{numeric}.mp3",
        AUDIO_ROOT / f"{numeric}.mp3",
    ]
    return next((str(path) for path in alternatives if path.is_file()), None)

selected = metadata[["TRACK_ID", "_key"]].copy()
selected["audio_path"] = metadata.apply(resolve_audio, axis=1)
missing = selected["audio_path"].isna().sum()
print(f"Selected: {len(selected):,} | Unique: {selected['_key'].nunique():,} | Missing audio: {missing}")
assert missing == 0


## 5. Harmony feature extractor


In [ ]:
def summarize(values):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0 or not np.isfinite(values).all():
        raise ValueError("Cannot summarize an empty or non-finite array")
    return float(values.mean()), float(values.std(ddof=0))


def extract_harmony_file(audio_path):
    y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True, duration=MAX_DURATION_SECONDS)
    if y.size == 0 or not np.isfinite(y).all():
        raise ValueError("Decoded waveform is empty or non-finite")

    duration = float(len(y) / sr)
    tuning = float(librosa.estimate_tuning(y=y, sr=sr, bins_per_octave=BINS_PER_OCTAVE))
    raw = librosa.feature.chroma_cqt(
        y=y, sr=sr, hop_length=HOP_LENGTH, n_chroma=N_CHROMA,
        bins_per_octave=BINS_PER_OCTAVE, n_octaves=N_OCTAVES,
        tuning=tuning, norm=None, threshold=0.0,
    )
    frames = raw.shape[1]
    rms = librosa.feature.rms(y=y, hop_length=HOP_LENGTH, center=True)[0]
    if len(rms) < frames:
        rms = np.pad(rms, (0, frames - len(rms)))
    else:
        rms = rms[:frames]

    threshold = max(ABSOLUTE_RMS_THRESHOLD, RELATIVE_RMS_THRESHOLD * float(rms.max(initial=0.0)))
    sums = raw.sum(axis=0)
    valid = (rms > threshold) & np.isfinite(sums) & (sums > np.finfo(np.float32).eps)
    if int(valid.sum()) < 2:
        raise ValueError("Fewer than two valid tonal frames")

    chroma = np.zeros_like(raw, dtype=np.float32)
    chroma[:, valid] = raw[:, valid] / sums[valid]
    valid_chroma = chroma[:, valid]
    tonnetz_full = librosa.feature.tonnetz(chroma=chroma, sr=sr)
    valid_tonnetz = tonnetz_full[:, valid]

    features = {}
    for index, name in enumerate(PITCH_NAMES):
        features[f"chroma_{name}_mean"], features[f"chroma_{name}_std"] = summarize(valid_chroma[index])
    for index in range(6):
        features[f"tonnetz_{index+1:02d}_mean"], features[f"tonnetz_{index+1:02d}_std"] = summarize(valid_tonnetz[index])

    concentration = valid_chroma.max(axis=0)
    entropy = -(valid_chroma * np.log(valid_chroma + 1e-12)).sum(axis=0) / math.log(12.0)
    valid_pairs = valid[:-1] & valid[1:]
    if not np.any(valid_pairs):
        raise ValueError("No consecutive valid tonal frame pairs")
    chroma_flux = np.linalg.norm(np.diff(chroma, axis=1)[:, valid_pairs], axis=0)
    tonnetz_movement = np.linalg.norm(np.diff(tonnetz_full, axis=1)[:, valid_pairs], axis=0)

    features["tonal_concentration_mean"], features["tonal_concentration_std"] = summarize(concentration)
    features["chroma_entropy_mean"], features["chroma_entropy_std"] = summarize(entropy)
    features["chroma_flux_mean"], features["chroma_flux_std"] = summarize(chroma_flux)
    features["tonnetz_movement_mean"], features["tonnetz_movement_std"] = summarize(tonnetz_movement)
    features["valid_tonal_ratio"] = float(valid.mean())

    values = np.asarray([features[name] for name in FEATURE_COLUMNS], dtype=np.float64)
    if not np.isfinite(values).all():
        raise ValueError("Extracted harmony features contain non-finite values")
    return features, {
        "analysis_duration_sec": duration,
        "total_chroma_frames": int(frames),
        "valid_chroma_frames": int(valid.sum()),
        "tuning_bins": tuning,
    }


## 6. Two-track smoke test and timing sample


In [ ]:
smoke_rows = selected.iloc[[0, len(selected) // 2]].to_dict("records")
smoke_started = time.time()
for row in smoke_rows:
    features, diagnostics = extract_harmony_file(row["audio_path"])
    assert len(features) == 45 and all(np.isfinite(list(features.values())))
print(f"Smoke test passed: 2 tracks in {time.time()-smoke_started:.1f} seconds")


## 7. Resumable parallel extraction


In [ ]:
META_COLUMNS = [
    "TRACK_ID", "status", "audio_path", "analysis_start_sec", "analysis_end_sec",
    "analysis_duration_sec", "sample_rate", "hop_length", "extractor",
    "total_chroma_frames", "valid_chroma_frames", "tuning_bins", "elapsed_sec",
    "error_type", "error_message",
]
ALL_COLUMNS = META_COLUMNS + FEATURE_COLUMNS

def load_records():
    if not CHECKPOINT_CSV.is_file():
        return {}
    frame = pd.read_csv(CHECKPOINT_CSV, dtype={"TRACK_ID": str}, keep_default_na=True)
    return {canonical_track_id(row["TRACK_ID"]): row.to_dict() for _, row in frame.iterrows()}

def save_checkpoint(records):
    frame = pd.DataFrame(records.values())
    for column in ALL_COLUMNS:
        if column not in frame:
            frame[column] = np.nan
    order = {key: i for i, key in enumerate(selected["_key"])}
    frame["_order"] = frame["TRACK_ID"].map(canonical_track_id).map(order)
    frame = frame.sort_values("_order").drop(columns="_order").loc[:, ALL_COLUMNS]
    temporary = CHECKPOINT_CSV.with_suffix(".tmp.csv")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, CHECKPOINT_CSV)
    return frame

def backup_outputs():
    run_quiet_rclone(["copy", str(OUTPUT_ROOT), f"{REMOTE_DATA}/harmony_features", "--exclude", "*.tmp.csv", "--checkers", "8"])

def process_track(row):
    started = time.time()
    base = {
        "TRACK_ID": row["TRACK_ID"], "audio_path": row["audio_path"],
        "analysis_start_sec": 0.0, "sample_rate": SAMPLE_RATE,
        "hop_length": HOP_LENGTH, "extractor": "librosa_chroma_cqt_first4min_v1",
    }
    try:
        features, diagnostics = extract_harmony_file(row["audio_path"])
        return {
            **base, **diagnostics, **features, "status": "ok",
            "analysis_end_sec": diagnostics["analysis_duration_sec"],
            "error_type": "", "error_message": "", "elapsed_sec": time.time() - started,
        }
    except Exception as error:
        return {
            **base, "status": "error", "error_type": type(error).__name__,
            "error_message": str(error)[:1000], "elapsed_sec": time.time() - started,
        }


records = load_records()
pending = []
for row in selected.to_dict("records"):
    previous = records.get(row["_key"])
    if previous is None or (RETRY_FAILED_TRACKS and previous.get("status") != "ok"):
        pending.append(row)
if LIMIT_TRACKS is not None:
    pending = pending[:int(LIMIT_TRACKS)]

print(f"Checkpoint rows: {len(records):,} | Pending: {len(pending):,} | Workers: {N_WORKERS}")
run_started = time.time()
completed_run = 0
last_backup = 0

with Parallel(n_jobs=N_WORKERS, backend="loky", batch_size=1, pre_dispatch=N_WORKERS) as parallel:
    for start in range(0, len(pending), PARALLEL_BATCH_SIZE):
        batch = pending[start:start + PARALLEL_BATCH_SIZE]
        results = parallel(delayed(process_track)(row) for row in batch)
        for record in results:
            records[canonical_track_id(record["TRACK_ID"])] = record
        completed_run += len(results)
        checkpoint = save_checkpoint(records)

        if completed_run - last_backup >= DRIVE_BACKUP_EVERY:
            backup_outputs()
            last_backup = completed_run

        # One concise line per 250 tracks or at completion; no per-file output.
        if completed_run % 250 < len(results) or completed_run == len(pending):
            elapsed_h = (time.time() - run_started) / 3600
            rate = completed_run / max(time.time() - run_started, 1)
            eta_h = (len(pending) - completed_run) / max(rate, 1e-9) / 3600
            ok = sum(str(v.get("status")) == "ok" for v in records.values())
            errors = len(records) - ok
            print(f"Progress {completed_run:,}/{len(pending):,} | saved={len(records):,} | ok={ok:,} | errors={errors:,} | elapsed={elapsed_h:.2f}h | ETA={eta_h:.2f}h")
            gc.collect()

checkpoint = save_checkpoint(records)
backup_outputs()
print(f"Extraction pass complete in {(time.time()-run_started)/3600:.2f} hours")


## 8. Final validation and export


In [ ]:
checkpoint = pd.read_csv(CHECKPOINT_CSV, dtype={"TRACK_ID": str}, keep_default_na=True)
ok = checkpoint.loc[checkpoint["status"] == "ok"].copy()
errors = checkpoint.loc[checkpoint["status"] != "ok"].copy()

numeric = ok[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
finite = np.isfinite(numeric.to_numpy(dtype=float))

print(f"Rows={len(checkpoint):,} | OK={len(ok):,} | Errors={len(errors):,} | Unique IDs={checkpoint['TRACK_ID'].nunique():,}")
print(f"Missing feature cells={int(numeric.isna().sum().sum()):,} | Non-finite cells={int((~finite).sum()):,}")

assert len(checkpoint) == 7_324
assert checkpoint["TRACK_ID"].nunique() == 7_324
assert len(errors) == 0
assert finite.all()

checkpoint.to_csv(FINAL_RAW_CSV, index=False)
ok[["TRACK_ID"] + FEATURE_COLUMNS].to_csv(FINAL_CLEAN_CSV, index=False)
errors.to_csv(ERROR_CSV, index=False)

config = {
    "schema_version": "harmony_summary_first4min_v1",
    "tracks": 7_324, "features": FEATURE_COLUMNS,
    "sample_rate": SAMPLE_RATE, "hop_length": HOP_LENGTH,
    "max_duration_seconds": MAX_DURATION_SECONDS,
    "extractor": "librosa.feature.chroma_cqt",
    "bins_per_octave": BINS_PER_OCTAVE, "n_octaves": N_OCTAVES,
    "workers": N_WORKERS,
}
CONFIG_JSON.write_text(json.dumps(config, indent=2) + "\n")
backup_outputs()
print("FINAL VALIDATION: PASS")
print("Clean CSV:", FINAL_CLEAN_CSV)
